# Preprocessing Dataset Merge Baru — ABSA Hotel Santika
## Pipeline (10 tahap)

| # | Tahap |
|---|---|
| 1 | Raw merged dataset |
| 2 | Hapus duplikat exact |
| 3 | Filter review < 20 karakter |
| 4 | Normalisasi teks dasar (emoji, whitespace, karakter kontrol, smart quotes) + filter ulang |
| 5 | Case folding (lowercase) |
| 6 | Normalisasi slang & singkatan |
| 7 | Normalisasi karakter berulang (≥3 → 1) |
| 8 | Pembersihan tanda baca |
| 9 | Hapus duplikat pasca-normalisasi |
| 10 | Filter panjang final (≥ 20 karakter) |

## Input
`dataset_absa_santika_merged_raw.csv`
(kolom: `ID_Review, Platform, Nama_Hotel, Review_Date, Text_Review`)

## Output
`dataset_absa_santika_clean.csv`
(kolom: `review_id, platform, hotel_name, text_review, text_review_original, date`)

Format output sama persis dengan dataset clean historis untuk kompatibilitas.

## 1. Import dan Konfigurasi

In [1]:
from pathlib import Path
import re
import json
import pandas as pd

MIN_LENGTH = 20          # sama dengan pipeline historis
OUTPUT_FILE = 'dataset_absa_santika_clean_v2.csv'
AUDIT_FILE  = 'preprocessing_audit_v2.csv'
print('MIN_LENGTH:', MIN_LENGTH)

MIN_LENGTH: 20


## 2. Menemukan dan Memuat Dataset Merge Baru

In [2]:
FILENAME = 'dataset_absa_santika_merged_raw.csv'

def resolve_input():
    candidates = [Path(FILENAME), Path('Merge') / FILENAME]
    ki = Path('/kaggle/input')
    if ki.exists():
        candidates.extend(sorted(ki.rglob(FILENAME)))
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(
        f'{FILENAME} tidak ditemukan. Upload sebagai input Kaggle.')

INPUT_PATH = resolve_input()
raw = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
print('Sumber   :', INPUT_PATH)
print('Raw rows :', len(raw))
print('Kolom    :', list(raw.columns))
raw.head(3)

Sumber   : /kaggle/input/datasets/vince0014/dataset-absa-santika-merged-raw/dataset_absa_santika_merged_raw.csv
Raw rows : 18297
Kolom    : ['ID_Review', 'Platform', 'Nama_Hotel', 'Review_Date', 'Text_Review']


,ID_Review,Platform,Nama_Hotel,Review_Date,Text_Review
0,REV-000001,Agoda,Hotel Santika Bandung,2026-05-17,I always stay at Santika when in Bandung. Many...
1,REV-000002,Agoda,Hotel Santika Bandung,2026-05-11,"You can walk directly to BIP, because the loca..."
2,REV-000003,Agoda,Hotel Santika Bandung,2026-05-05,The location is strategic at the city centre. ...


## 3. Penyesuaian Kolom (Mapping ke Format Historis)

In [3]:
# Dataset merge baru pakai nama kolom berbeda dari pipeline historis.
# Mapping: ID_Review->review_id, Platform->platform, Nama_Hotel->hotel_name,
#          Review_Date->date, Text_Review->text_review
df = raw.rename(columns={
    'ID_Review'  : 'review_id',
    'Platform'   : 'platform',
    'Nama_Hotel' : 'hotel_name',
    'Review_Date': 'date',
    'Text_Review': 'text_review',
})
# Simpan teks asli sebelum normalisasi (untuk audit, sama dengan historis)
df['text_review_original'] = df['text_review'].copy()
df['text_review'] = df['text_review'].astype(str).fillna('')
print('Kolom setelah mapping:', list(df.columns))

Kolom setelah mapping: ['review_id', 'platform', 'hotel_name', 'date', 'text_review', 'text_review_original']


## 4. Install Dependensi Translate

`lingua-language-detector` (deteksi bahasa akurat untuk teks pendek) dan
`deep-translator` (Google Translate gratis). Best-effort; diinstall sekali.

In [4]:
import subprocess, sys
for pkg in ['lingua-language-detector', 'deep-translator']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                   check=False, timeout=300)
print('Dependensi siap.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.4 MB/s eta 0:00:00
Dependensi siap.


## 5. Deteksi Bahasa (lingua)

Deteksi bahasa setiap review menggunakan `lingua` dengan whitelist 19 bahasa
yang realistis untuk hotel di Indonesia. Threshold confidence 0.50; teks
pendek/tidak yakin → `unknown`. Malay dipetakan ke `id` (sangat mirip Indonesia).

Output kolom `original_language` (kode ISO 639-1).

In [5]:
import time
from lingua import Language, LanguageDetectorBuilder

ALLOWED_LANGS = [
    Language.INDONESIAN, Language.MALAY, Language.ENGLISH,
    Language.JAPANESE, Language.KOREAN, Language.CHINESE,
    Language.ARABIC, Language.TAGALOG, Language.THAI,
    Language.VIETNAMESE, Language.DUTCH, Language.FRENCH,
    Language.GERMAN, Language.SPANISH, Language.ITALIAN,
    Language.PORTUGUESE, Language.RUSSIAN, Language.HINDI,
    Language.TURKISH,
]

LANG_DETECTOR = (
    LanguageDetectorBuilder
    .from_languages(*ALLOWED_LANGS)
    .with_preloaded_language_models()
    .with_minimum_relative_distance(0.0)
    .build()
)

LANG_TO_CODE = {
    Language.INDONESIAN: 'id', Language.MALAY: 'id', Language.ENGLISH: 'en',
    Language.JAPANESE: 'ja', Language.KOREAN: 'ko', Language.CHINESE: 'zh',
    Language.ARABIC: 'ar', Language.TAGALOG: 'tl', Language.THAI: 'th',
    Language.VIETNAMESE: 'vi', Language.DUTCH: 'nl', Language.FRENCH: 'fr',
    Language.GERMAN: 'de', Language.SPANISH: 'es', Language.ITALIAN: 'it',
    Language.PORTUGUESE: 'pt', Language.RUSSIAN: 'ru', Language.HINDI: 'hi',
    Language.TURKISH: 'tr',
}

def detect_language(text):
    if not isinstance(text, str): return 'unknown'
    text_clean = text.strip()
    if len(text_clean) < 10: return 'unknown'
    try:
        conf = LANG_DETECTOR.compute_language_confidence_values(text_clean)
        if not conf: return 'unknown'
        top = conf[0]
        if top.value < 0.50: return 'unknown'
        return LANG_TO_CODE.get(top.language, 'unknown')
    except Exception:
        return 'unknown'

print('Mendeteksi bahasa (estimasi ~1-2 menit)...')
start = time.time()
df['original_language'] = df['text_review'].apply(detect_language)
print(f'Selesai dalam {time.time()-start:.1f} detik')

lang_dist = df['original_language'].value_counts()
LABEL_MAP = {
    'id':'Indonesia','en':'English','ja':'Japanese','ko':'Korean','zh':'Chinese',
    'ar':'Arabic','tl':'Tagalog','th':'Thai','vi':'Vietnamese','nl':'Dutch',
    'fr':'French','de':'German','es':'Spanish','it':'Italian','pt':'Portuguese',
    'ru':'Russian','hi':'Hindi','tr':'Turkish','unknown':'Tdk terdeteksi/pendek',
}
for lang, count in lang_dist.items():
    print(f'  {lang:>7s} ({LABEL_MAP.get(lang,lang)}): {count:,} ({count/len(df)*100:.1f}%)')
print('\nBreakdown per platform:')
display(pd.crosstab(df['platform'], df['original_language'], margins=True))

Mendeteksi bahasa (estimasi ~1-2 menit)...
Selesai dalam 11.1 detik
       id (Indonesia): 14,206 (77.6%)
       en (English): 2,275 (12.4%)
  unknown (Tdk terdeteksi/pendek): 1,633 (8.9%)
       ja (Japanese): 64 (0.3%)
       ko (Korean): 28 (0.2%)
       tl (Tagalog): 21 (0.1%)
       nl (Dutch): 18 (0.1%)
       fr (French): 13 (0.1%)
       ar (Arabic): 12 (0.1%)
       de (German): 11 (0.1%)
       zh (Chinese): 7 (0.0%)
       es (Spanish): 3 (0.0%)
       th (Thai): 2 (0.0%)
       it (Italian): 2 (0.0%)
       ru (Russian): 2 (0.0%)

Breakdown per platform:


original_language,ar,de,en,es,fr,id,it,ja,ko,nl,ru,th,tl,unknown,zh,All
platform,,,,,,,,,,,,,,,,
Agoda,11,11,2050,0,11,3720,1,61,28,15,2,2,3,748,7,6670
Tiket,1,0,218,3,2,2080,0,3,0,3,0,0,12,442,0,2764
Traveloka,0,0,7,0,0,8406,1,0,0,0,0,0,6,443,0,8863
All,12,11,2275,3,13,14206,2,64,28,18,2,2,21,1633,7,18297


## 6. Translate Semua Review Non-Indonesia → Indonesia

Semua review bukan `id` dan bukan `unknown` diterjemahkan ke Indonesia via
`deep-translator` (Google Translate). Teks asli sudah disimpan ke
`text_review_original`. Checkpoint disimpan setiap 200 review agar tahan disconnect.

In [6]:
from deep_translator import GoogleTranslator
from pathlib import Path

CHECKPOINT_PATH = Path('/kaggle/working/_translation_checkpoint.csv') \
    if Path('/kaggle/working').exists() else Path('_translation_checkpoint.csv')

LANGS_TO_TRANSLATE = [l for l in df['original_language'].unique()
                      if l not in ('id', 'unknown')]
needs_translation = df['original_language'].isin(LANGS_TO_TRANSLATE)
total_to_translate = int(needs_translation.sum())

print(f'Review akan ditranslate: {total_to_translate:,}')
print(f'Bahasa: {sorted(LANGS_TO_TRANSLATE)}')

def translate_one(text, src_lang, target='id'):
    if not text or len(str(text).strip()) == 0: return text
    text = str(text)[:4900]
    try:
        r = GoogleTranslator(source=src_lang, target=target).translate(text)
        return r if r else text
    except Exception:
        try:
            r = GoogleTranslator(source='auto', target=target).translate(text)
            return r if r else text
        except Exception:
            return text

def translate_indices(df, indices, batch_delay=0.3):
    results = {}
    total = len(indices)
    for i, idx in enumerate(indices, 1):
        text = df.at[idx, 'text_review']
        src = df.at[idx, 'original_language']
        gt_src = src if src != 'unknown' else 'auto'
        results[idx] = translate_one(text, gt_src, target='id')
        if i % 10 == 0 or i == total:
            print(f'  Translated: {i:,}/{total:,} ({i/total*100:.1f}%)', end='\r')
        time.sleep(batch_delay)
    print()
    return results

if total_to_translate > 0:
    en_indices = df[needs_translation].index.tolist()

    # Cek checkpoint
    if CHECKPOINT_PATH.exists():
        df_ckpt = pd.read_csv(CHECKPOINT_PATH, encoding='utf-8-sig')
        translated_map = dict(zip(df_ckpt['index'].astype(int), df_ckpt['translated']))
        remaining = [i for i in en_indices if i not in translated_map]
        print(f'Checkpoint ditemukan: {len(translated_map):,} sudah, {len(remaining):,} sisa')
    else:
        translated_map = {}
        remaining = en_indices

    if remaining:
        print(f'\nMenerjemahkan {len(remaining):,} review...')
        CKPT_EVERY = 200
        for chunk_start in range(0, len(remaining), CKPT_EVERY):
            chunk = remaining[chunk_start:chunk_start + CKPT_EVERY]
            translated_map.update(translate_indices(df, chunk, batch_delay=0.3))
            ck = pd.DataFrame([{'index': k, 'translated': v}
                                for k, v in translated_map.items()])
            ck.to_csv(CHECKPOINT_PATH, index=False, encoding='utf-8-sig')
            print(f'  Checkpoint disimpan ({len(translated_map):,} total)')

    for idx, t in translated_map.items():
        if idx in df.index:
            df.at[idx, 'text_review'] = t

    print(f'\nTranslate selesai: {len(translated_map):,} review diterjemahkan.')
else:
    print('Tidak ada review non-Indonesia, skip translate.')

Review akan ditranslate: 2,458
Bahasa: ['ar', 'de', 'en', 'es', 'fr', 'it', 'ja', 'ko', 'nl', 'ru', 'th', 'tl', 'zh']

Menerjemahkan 2,458 review...
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (200 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (400 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (600 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (800 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (1,000 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (1,200 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (1,400 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (1,600 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (1,800 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (2,000 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (2,200 total)
  Translated: 200/200 (100.0%)
  Checkpoint disimpan (2,400 total)
  Translated: 58/58 (100.0%)
  Checkpoint disimpan (2,4

## 7. Re-detect + Rollback (Koreksi Salah Translate)

Re-detect bahasa pada teks asli (`text_review_original`) dengan threshold lebih
longgar (0.30) + heuristik kata kunci Indonesia. Review yang awalnya salah
ditranslate (sebenarnya Indonesia) di-rollback ke teks asli.

In [7]:
import re as _re

ID_KEYWORDS = {
    'yang','saya','tidak','sangat','dengan','untuk','sudah','juga',
    'bagus','enak','ramah','bersih','nyaman','kamar','hotel','staff',
    'staf','pelayanan','sarapan','kolam','pemandangan','lokasi','mantap',
    'menyenangkan','membantu','gak','aja','banget','lagi','kalo','kalau',
    'biasa','cukup','kurang','lebih','banyak','sekali','kembali','baik',
    'baru','lama','tempat','dekat','jauh','breakfast',
}

def is_indonesian_keywords(text, min_hits=2):
    if not isinstance(text, str): return False
    words = _re.findall(r'\b[a-zA-Z]+\b', text.lower())
    return sum(1 for w in words if w in ID_KEYWORDS) >= min_hits

def detect_lang_v2(text):
    if not isinstance(text, str): return 'unknown'
    t = text.strip()
    if len(t) < 10:
        return 'id' if is_indonesian_keywords(t, 1) else 'unknown'
    try:
        conf = LANG_DETECTOR.compute_language_confidence_values(t)
        if not conf: return 'id' if is_indonesian_keywords(t) else 'unknown'
        top = conf[0]
        top_code = LANG_TO_CODE.get(top.language, 'unknown')
        if top.value >= 0.30:
            if top_code == 'tl' and is_indonesian_keywords(t): return 'id'
            return top_code
        if is_indonesian_keywords(t): return 'id'
        if top_code == 'id' and top.value >= 0.15: return 'id'
        return 'unknown'
    except Exception:
        return 'id' if is_indonesian_keywords(t) else 'unknown'

print('Re-detect bahasa (threshold 0.30 + heuristik ID)...')
df['original_language_new'] = df['text_review_original'].apply(detect_lang_v2)
changed = int((df['original_language'] != df['original_language_new']).sum())
print(f'Label berubah: {changed:,}')

# Apply label baru
df['original_language'] = df['original_language_new']
df = df.drop(columns=['original_language_new'])

# Rollback teks yang salah ditranslate (sekarang id/unknown tapi teksnya sudah berbeda)
needs_rollback = (
    df['original_language'].isin(['id', 'unknown'])
    & (df['text_review'] != df['text_review_original'])
)
n_rollback = int(needs_rollback.sum())
if n_rollback > 0:
    df.loc[needs_rollback, 'text_review'] = df.loc[needs_rollback, 'text_review_original']
    print(f'Rollback: {n_rollback:,} review dikembalikan ke teks asli')
else:
    print('Tidak ada yang perlu di-rollback.')

# Translate ulang yang baru terdeteksi non-id
LANGS_NEW = [l for l in df['original_language'].unique() if l not in ('id','unknown')]
needs_new = (
    df['original_language'].isin(LANGS_NEW)
    & (df['text_review'] == df['text_review_original'])
)
n_new = int(needs_new.sum())
if n_new > 0:
    print(f'\nTranslate ulang {n_new:,} review baru terdeteksi...')
    for idx in df[needs_new].index:
        t = translate_one(df.at[idx,'text_review_original'],
                          df.at[idx,'original_language'], target='id')
        df.at[idx,'text_review'] = t
        time.sleep(0.3)
    print(f'Selesai: {n_new:,} diterjemahkan.')
else:
    print('Tidak ada review baru yang perlu ditranslate ulang.')

# Hapus checkpoint setelah selesai
if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print('Checkpoint dihapus.')

print('\nDistribusi bahasa FINAL:')
display(df['original_language'].value_counts().rename_axis('Bahasa').reset_index(name='Jumlah'))
n_translated = int((df['text_review'] != df['text_review_original']).sum())
print(f'Total ditranslate: {n_translated:,} | Bahasa asli: {len(df)-n_translated:,}')

Re-detect bahasa (threshold 0.30 + heuristik ID)...
Label berubah: 1,242
Rollback: 15 review dikembalikan ke teks asli

Translate ulang 252 review baru terdeteksi...
Selesai: 252 diterjemahkan.
Checkpoint dihapus.

Distribusi bahasa FINAL:


,Bahasa,Jumlah
0,id,15199
1,en,2469
2,unknown,406
3,ja,64
4,tl,29
5,ko,28
6,fr,22
7,nl,21
8,de,15
9,es,12


Total ditranslate: 2,681 | Bahasa asli: 15,616


## 8. Definisi Fungsi Normalisasi (Identik dengan Pipeline Historis)

In [8]:
EMOJI_PATTERN = re.compile(
    '['
    '\U0001F600-\U0001F64F'
    '\U0001F300-\U0001F5FF'
    '\U0001F680-\U0001F6FF'
    '\U0001F1E0-\U0001F1FF'
    '\U00002702-\U000027B0'
    '\U0001F900-\U0001F9FF'
    '\U00002600-\U000026FF'
    '\u200d\ufe0f'
    ']'
)
EMOJIS_TO_REMOVE = (
    '😀😃😄😁😆😅🤣😂🙂😊😍🥰😘😗😙😚😋😛😜🤪😝🤗🤭🤫🤔'
    '😐😑😶😏😒🙄😬😮😯😲😳🥺😢😭😤😠😡🤬😈👿👍👎👏🙏'
    '❤️💯⭐🌟✅❌✨🔥🥴💪😎'
)

def normalize_text(text):
    if not isinstance(text, str) or not text:
        return ''
    for e in EMOJIS_TO_REMOVE:
        text = text.replace(e, '')
    text = EMOJI_PATTERN.sub('', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u2014', ' - ').replace('\u2013', ' - ')
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\t+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

SLANG_DICT = {
    'yg':'yang','dgn':'dengan','dg':'dengan','utk':'untuk','krn':'karena',
    'tp':'tapi','tdk':'tidak','gak':'tidak','ga':'tidak','gk':'tidak',
    'nggak':'tidak','enggak':'tidak','trs':'terus','trus':'terus',
    'bgt':'banget','bngt':'banget','bkn':'bukan','blm':'belum','sdh':'sudah',
    'udh':'sudah','udah':'sudah','lg':'lagi','lgi':'lagi','jg':'juga',
    'jgn':'jangan','sm':'sama','dr':'dari','dlm':'dalam','dpt':'dapat',
    'hrs':'harus','spy':'supaya','ttg':'tentang','org':'orang','sy':'saya',
    'ak':'aku','kmu':'kamu','km':'kamu','bs':'bisa','cm':'cuma','kl':'kalau',
    'klo':'kalau','kalo':'kalau','mkn':'mungkin','mgkn':'mungkin',
    'emg':'memang','emang':'memang','bbrp':'beberapa','brg':'barang',
    'bgmn':'bagaimana','gmn':'gimana','dmn':'dimana','kmr':'kamar',
    'tmpt':'tempat','smua':'semua','bnr':'benar','bner':'benar','ckp':'cukup',
    'scr':'secara','trm':'terima','trmksh':'terima kasih','thx':'terima kasih',
    'thanks':'terima kasih','thank':'terima kasih','tq':'terima kasih',
    'makasih':'terima kasih','mksh':'terima kasih','mks':'terima kasih',
    'mantap':'bagus','mantapp':'bagus','mantapppp':'bagus','mantul':'bagus',
    'keren':'bagus','oke':'baik','okee':'baik','okeee':'baik','okeeee':'baik',
    'okey':'baik','ok':'baik','okay':'baik','okelah':'baik','okeh':'baik',
    'jelek':'buruk','jlk':'buruk','ancur':'buruk','parah':'buruk','zonk':'buruk',
    'bfast':'breakfast','b-fast':'breakfast','brekfas':'breakfast',
    'breskfast':'breakfast','breakfas':'breakfast',
    'good':'bagus','great':'bagus','nice':'bagus','bad':'buruk',
    'helpful':'membantu','friendly':'ramah','comfortable':'nyaman',
    'clean':'bersih','staff':'staf','ac':'air conditioner','wifi':'wi-fi',
    'wf':'wi-fi','tv':'televisi',
}

def normalize_slang(text):
    result = []
    for word in text.split():
        cw = re.sub(r'[.,!?;:]+$', '', word)
        trail = word[len(cw):]
        result.append(SLANG_DICT.get(cw, cw) + trail)
    return ' '.join(result)

def normalize_repeated_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)

def clean_punctuation(text):
    text = re.sub(r'([!?.]){2,}', r'\1', text)
    text = re.sub(r'[~*#@^&|\\{}\[\]<>]+', ' ', text)
    text = re.sub(r'\s*([.,!?;:])\s*', r'\1 ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

print('Fungsi normalisasi siap.')

Fungsi normalisasi siap.


## 9. Menjalankan Pipeline Preprocessing (10 Tahap)

In [9]:
audit = []
def log(step, before, after, desc):
    audit.append({'step':step,'rows_before':before,'rows_after':after,
                  'removed':before-after,'description':desc})
    print(f'[{step}] {before:,} -> {after:,} (hapus {before-after:,}) | {desc}')

log('1_raw', len(df), len(df), 'Raw merged dataset')

# Tahap 2: Hapus duplikat exact
b = len(df); df = df.drop_duplicates(subset='text_review', keep='first')
log('2_drop_exact_dup', b, len(df), 'Hapus duplikat exact text_review')

# Tahap 3: Filter review < MIN_LENGTH karakter
b = len(df); df = df[df['text_review'].astype(str).str.len() >= MIN_LENGTH]
log('3_filter_short', b, len(df), f'Hapus review < {MIN_LENGTH} karakter')

# Tahap 4: Normalisasi teks dasar + filter ulang
df['text_review'] = df['text_review'].apply(normalize_text)
b = len(df)
df = df[df['text_review'].str.strip().astype(bool)]
df = df[df['text_review'].str.len() >= MIN_LENGTH]
log('4_normalize_basic', b, len(df), 'Normalisasi teks dasar + filter kosong/pendek')

# Tahap 5: Case folding
b = len(df); df['text_review'] = df['text_review'].str.lower()
log('5_case_folding', b, len(df), 'Lowercase')

# Tahap 6: Normalisasi slang
b = len(df); df['text_review'] = df['text_review'].apply(normalize_slang)
log('6_slang', b, len(df), 'Normalisasi slang & singkatan')

# Tahap 7: Karakter berulang
b = len(df); df['text_review'] = df['text_review'].apply(normalize_repeated_chars)
log('7_repeat_char', b, len(df), 'Normalisasi karakter berulang ≥3 → 1')

# Tahap 8: Pembersihan tanda baca
b = len(df); df['text_review'] = df['text_review'].apply(clean_punctuation)
log('8_punctuation', b, len(df), 'Bersihkan tanda baca')

# Tahap 9: Hapus duplikat pasca-normalisasi
b = len(df); df = df.drop_duplicates(subset='text_review', keep='first')
log('9_drop_post_dup', b, len(df), 'Hapus duplikat pasca-normalisasi')

# Tahap 10: Filter panjang final
b = len(df); df = df[df['text_review'].str.strip().str.len() >= MIN_LENGTH]
log('10_final_filter', b, len(df), f'Filter final < {MIN_LENGTH} karakter')

print('\nSelesai. Baris bersih:', len(df))

[1_raw] 18,297 -> 18,297 (hapus 0) | Raw merged dataset
[2_drop_exact_dup] 18,297 -> 15,650 (hapus 2,647) | Hapus duplikat exact text_review
[3_filter_short] 15,650 -> 15,182 (hapus 468) | Hapus review < 20 karakter
[4_normalize_basic] 15,182 -> 15,169 (hapus 13) | Normalisasi teks dasar + filter kosong/pendek
[5_case_folding] 15,169 -> 15,169 (hapus 0) | Lowercase
[6_slang] 15,169 -> 15,169 (hapus 0) | Normalisasi slang & singkatan
[7_repeat_char] 15,169 -> 15,169 (hapus 0) | Normalisasi karakter berulang ≥3 → 1
[8_punctuation] 15,169 -> 15,169 (hapus 0) | Bersihkan tanda baca
[9_drop_post_dup] 15,169 -> 15,145 (hapus 24) | Hapus duplikat pasca-normalisasi
[10_final_filter] 15,145 -> 15,095 (hapus 50) | Filter final < 20 karakter

Selesai. Baris bersih: 15095


## 10. Menyusun Output Final

In [10]:
df = df.reset_index(drop=True)
df['review_id'] = range(1, len(df) + 1)
output_cols = ['review_id','platform','hotel_name','text_review','text_review_original','date']
clean = df[output_cols].copy()
print('Kolom output:', list(clean.columns))
print('\nDistribusi platform:')
display(clean['platform'].value_counts().rename_axis('platform').reset_index(name='jumlah'))
print('\nDistribusi hotel:')
display(clean['hotel_name'].value_counts().rename_axis('hotel_name').reset_index(name='jumlah'))
clean.head()

Kolom output: ['review_id', 'platform', 'hotel_name', 'text_review', 'text_review_original', 'date']

Distribusi platform:


,platform,jumlah
0,Traveloka,8825
1,Agoda,3895
2,Tiket,2375



Distribusi hotel:


,hotel_name,jumlah
0,Hotel Santika Megacity Bekasi,3198
1,Hotel Santika Bogor,2858
2,Hotel Santika Bandung,2650
3,Hotel Santika Depok,2498
4,Hotel Santika Cirebon,2048
5,Hotel Santika Tasikmalaya,1843


,review_id,platform,hotel_name,text_review,text_review_original,date
0,1,Agoda,Hotel Santika Bandung,saya selalu menginap di santika bila di bandun...,I always stay at Santika when in Bandung. Many...,2026-05-17
1,2,Agoda,Hotel Santika Bandung,"bisa jalan kaki langsung ke bip, karena lokasi...","You can walk directly to BIP, because the loca...",2026-05-11
2,3,Agoda,Hotel Santika Bandung,lokasinya strategis di pusat kota. makanan saa...,The location is strategic at the city centre. ...,2026-05-05
3,4,Agoda,Hotel Santika Bandung,hotel ini agak ketinggalan jaman. stafnya rama...,The hotel is a bit outdated. The staff are fri...,2026-03-31
4,5,Agoda,Hotel Santika Bandung,"pengalaman menginap yang sangat berkesan, pros...","A very memorable stay experience, the check-in...",2026-03-26


## 11. Tabel Audit Preprocessing

In [11]:
audit_df = pd.DataFrame(audit)
display(audit_df)

raw_n = int(audit[0]['rows_before'])
clean_n = len(clean)
print(f'\nRaw   : {raw_n:,}')
print(f'Clean : {clean_n:,}')
print(f'Hapus : {raw_n-clean_n:,} ({(raw_n-clean_n)/raw_n*100:.2f}%)')
print(f'(Historis: 17.868 -> 14.747 — perbedaan wajar karena ada tambahan Agoda Bekasi)')

,step,rows_before,rows_after,removed,description
0,1_raw,18297,18297,0,Raw merged dataset
1,2_drop_exact_dup,18297,15650,2647,Hapus duplikat exact text_review
2,3_filter_short,15650,15182,468,Hapus review < 20 karakter
3,4_normalize_basic,15182,15169,13,Normalisasi teks dasar + filter kosong/pendek
4,5_case_folding,15169,15169,0,Lowercase
5,6_slang,15169,15169,0,Normalisasi slang & singkatan
6,7_repeat_char,15169,15169,0,Normalisasi karakter berulang ≥3 → 1
7,8_punctuation,15169,15169,0,Bersihkan tanda baca
8,9_drop_post_dup,15169,15145,24,Hapus duplikat pasca-normalisasi
9,10_final_filter,15145,15095,50,Filter final < 20 karakter



Raw   : 18,297
Clean : 15,095
Hapus : 3,202 (17.50%)
(Historis: 17.868 -> 14.747 — perbedaan wajar karena ada tambahan Agoda Bekasi)


## 12. Menyimpan Output

In [12]:
out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
clean_path = out_dir / OUTPUT_FILE
audit_path = out_dir / AUDIT_FILE

clean.to_csv(clean_path, index=False, encoding='utf-8-sig')
audit_df.to_csv(audit_path, index=False, encoding='utf-8-sig')

manifest = {
    'input': str(INPUT_PATH),
    'output': str(clean_path),
    'raw_rows': raw_n,
    'clean_rows': clean_n,
    'removed_rows': raw_n - clean_n,
    'minimum_review_length': MIN_LENGTH,
    'pipeline': 'identical to reconstruct_preprocessing_pipeline.py',
}
(out_dir / 'preprocessing_manifest_v2.json').write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

print('Output tersimpan:')
print(' -', clean_path)
print(' -', audit_path)
print(' - preprocessing_manifest_v2.json')

Output tersimpan:
 - /kaggle/working/dataset_absa_santika_clean_v2.csv
 - /kaggle/working/preprocessing_audit_v2.csv
 - preprocessing_manifest_v2.json
